1. Gerekli Kütüphaneler ve Temizlenmiş Verinin Yüklenmesi

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import os
import gc

print("Temizlenmiş parçalı veriler okunuyor...")
df = pd.read_parquet('../data/cleaned_parquets/')

print("\nVeri setindeki tüm sınıflar ve benzersiz etiketler:")
sifir_görünüm = df['Label'].value_counts()
print(sifir_görünüm)

Temizlenmiş parçalı veriler okunuyor...

Veri setindeki tüm sınıflar ve benzersiz etiketler:
Label
Benign                      9686601
DDoS attacks-LOIC-HTTP       575012
DDOS attack-HOIC             198861
DoS attacks-Hulk             145199
Bot                          144535
Infilteration                118483
SSH-Bruteforce                94048
DoS attacks-GoldenEye         41392
DoS attacks-Slowloris          9712
DDOS attack-LOIC-UDP           1730
Brute Force -Web                568
Brute Force -XSS                229
SQL Injection                    85
DoS attacks-SlowHTTPTest         55
FTP-BruteForce                   53
Name: count, dtype: int64


2. Stratejik Etiket Gruplama ve Zero-Day Seçimi

In [2]:
# 1. Zero-Day adayı olacak saldırıları seçiyoruz (Eğitim setinden tamamen gizlenecekler)
# Üstteki hücrenin çıktısına göre isimleri birebir eşleşecek şekilde düzenleyebilirsin
zero_day_attacks = ['Bot', 'Infiltration', 'Infilteration'] 

# 2. Maskeleme yöntemiyle veriyi ayır
zero_day_mask = df['Label'].isin(zero_day_attacks)

# Sıfırıncı Gün Verisi (Sadece testte kullanılacak)
zero_day_data = df[zero_day_mask]

# Bilinen Veri (Eğitim ve normal test testlerinde kullanılacak: Benign + DoS, DDoS vb.)
known_data = df[~zero_day_mask]

print("--- GRUPLAMA ÖZETİ ---")
print(f"Eğitim ve Normal Test için kullanılacak (Bilinen) satır sayısı: {known_data.shape[0]}")
print(f"Zero-Day (Sıfırıncı Gün) simülasyonu için gizlenen satır sayısı: {zero_day_data.shape[0]}")

--- GRUPLAMA ÖZETİ ---
Eğitim ve Normal Test için kullanılacak (Bilinen) satır sayısı: 10753545
Zero-Day (Sıfırıncı Gün) simülasyonu için gizlenen satır sayısı: 263018


3. Girdiler (X) ve Hedeflerin (y) Anomali Formatına Çevrilmesi

In [3]:
def prepare_features_and_labels(data_frame):
    # Girdi özellikleri (Label sütunu hariç her şey)
    X = data_frame.drop(columns=['Label'])
    
    # Hedef Değişken: Benign ise 0, herhangi bir saldırı ise 1
    y = (data_frame['Label'] != 'Benign').astype(int)
    
    return X, y

print("Özellikler ve hedef değişkenler ayrıştırılıyor...")
X_known, y_known = prepare_features_and_labels(known_data)
X_zeroday, y_zeroday = prepare_features_and_labels(zero_day_data)

Özellikler ve hedef değişkenler ayrıştırılıyor...


4. Train/Test Split (Bölme) Stratejisinin Uygulanması

In [4]:
# 1. Bilinen veriyi %80 Train, %20 Test olacak şekilde bölüyoruz
X_train, X_test_known, y_train, y_test_known = train_test_split(
    X_known, y_known, 
    test_size=0.20, 
    random_state=42, 
    stratify=y_known # Sınıf oranlarını korumak için hayati öneme sahip
)

# 2. Gizlediğimiz Zero-Day verilerini Test setinin üzerine ekliyoruz (Birleştirme)
X_test = pd.concat([X_test_known, X_zeroday], ignore_index=True)
y_test = pd.concat([y_test_known, y_zeroday], ignore_index=True)

print("--- NİHAİ SET BOYUTLARI ---")
print(f"Eğitim Seti (X_train) -> Boyut: {X_train.shape} | Saldırı Oranı: %{round(y_train.mean()*100, 2)}")
print(f"Test Seti (X_test)   -> Boyut: {X_test.shape} | Saldırı Oranı: %{round(y_test.mean()*100, 2)}")
print(f"  └─ Test setindeki net Zero-Day satırı sayısı: {X_zeroday.shape[0]}")

--- NİHAİ SET BOYUTLARI ---
Eğitim Seti (X_train) -> Boyut: (8602836, 77) | Saldırı Oranı: %9.92
Test Seti (X_test)   -> Boyut: (2413727, 77) | Saldırı Oranı: %19.74
  └─ Test setindeki net Zero-Day satırı sayısı: 263018


5. Setlerin data/splits/ Altına Kaydedilmesi

In [5]:
output_dir = '../data/splits'
os.makedirs(output_dir, exist_ok=True)

print("Setler diske kaydediliyor, lütfen bekleyin...")

X_train.to_parquet(f'{output_dir}/X_train.parquet', index=False)
X_test.to_parquet(f'{output_dir}/X_test.parquet', index=False)

# y serilerini DataFrame'e dönüştürerek parquet formatında kaydediyoruz
pd.DataFrame(y_train, columns=['Label']).to_parquet(f'{output_dir}/y_train.parquet', index=False)
pd.DataFrame(y_test, columns=['Label']).to_parquet(f'{output_dir}/y_test.parquet', index=False)

print(f"Başarılı! Tüm veri setleri '{output_dir}/' klasörüne kaydedildi.")

# RAM temizliği
del X_known, y_known, X_zeroday, y_zeroday, X_test_known, y_test_known
gc.collect()

Setler diske kaydediliyor, lütfen bekleyin...
Başarılı! Tüm veri setleri '../data/splits/' klasörüne kaydedildi.


95